# 03 — Tutor-response evidence notebook

This notebook prepares a tutor-facing interpretation of the experimental results.

It converts the metrics into direct answers to the methodological questions:

- What is learned if models perform equally well on literal and metaphorical sentences?
- What is learned if they perform better on one than the other?
- What does the weak-schema control condition reveal?
- Does the image-schema layer add value beyond a good ordinary paraphrase?
- What does the v2 abstention mechanism contribute?

In [1]:
from __future__ import annotations

import json
import math
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Run these notebooks from the project notebooks/ directory.
PROJECT_ROOT = Path("..").resolve()
DATA_DIR = PROJECT_ROOT / "data"
OUTPUTS_DIR = DATA_DIR / "outputs"

RAW_PATH = OUTPUTS_DIR / "raw_responses.jsonl"
PARSED_PATH = OUTPUTS_DIR / "parsed_responses.jsonl"
GOLD_PATH = DATA_DIR / "gold" / "sentences_v1.jsonl"
COST_PATH = OUTPUTS_DIR / "cost_log.jsonl"

def read_jsonl(path: Path) -> pd.DataFrame:
    records = []
    if not path.exists():
        raise FileNotFoundError(f"File not found: {path}")
    with path.open("r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, start=1):
            if line.strip():
                try:
                    records.append(json.loads(line))
                except json.JSONDecodeError as exc:
                    raise ValueError(f"Invalid JSON in {path} at line {line_no}: {exc}") from exc
    return pd.DataFrame(records)

def safe_read_jsonl(path: Path) -> pd.DataFrame:
    return read_jsonl(path) if path.exists() else pd.DataFrame()

def clean_label(value):
    if pd.isna(value):
        return None
    return str(value).strip()

def norm_prompt_generation(prompt_id):
    if pd.isna(prompt_id):
        return "unknown"
    prompt_id = str(prompt_id)
    if "_v2_" in prompt_id or prompt_id.endswith("_v2"):
        return "v2_abstention"
    if "_v1" in prompt_id:
        return "v1"
    return "unknown"

def safe_accuracy(df: pd.DataFrame, pred_col: str, gold_col: str):
    if df.empty or pred_col not in df.columns or gold_col not in df.columns:
        return None
    sub = df[df[pred_col].notna() & df[gold_col].notna()].copy()
    if sub.empty:
        return None
    return float((sub[pred_col].astype(str) == sub[gold_col].astype(str)).mean())

def safe_rate(series):
    if series is None or len(series) == 0:
        return None
    return float(pd.Series(series).mean())

def pct(x, digits=1):
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return "NA"
    return f"{100*x:.{digits}f}%"

def add_derived_columns(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    if "prompt_id" in out.columns:
        out["prompt_generation"] = out["prompt_id"].map(norm_prompt_generation)
        out["uses_abstention_gate"] = out["prompt_id"].astype(str).str.contains("abstention", case=False, na=False)
    else:
        out["prompt_generation"] = "unknown"
        out["uses_abstention_gate"] = False

    if "sentence_type" in out.columns:
        out["gold_schema_present"] = np.where(out["sentence_type"].eq("control_weak_schema"), "no", "yes")
    else:
        out["gold_schema_present"] = np.nan

    if "main_image_schema" in out.columns:
        out["predicted_schema_present_from_schema"] = np.where(out["main_image_schema"].eq("NONE"), "no", "yes")
    else:
        out["predicted_schema_present_from_schema"] = np.nan

    if "schema_present" not in out.columns:
        out["schema_present"] = out["predicted_schema_present_from_schema"]

    if "literal_or_metaphorical" not in out.columns:
        out["literal_or_metaphorical"] = np.nan

    if "expected_literal_or_metaphorical" not in out.columns:
        out["expected_literal_or_metaphorical"] = np.nan

    if "main_image_schema" not in out.columns:
        out["main_image_schema"] = np.nan

    if "expected_schema_primary" not in out.columns:
        out["expected_schema_primary"] = np.nan

    out["is_control"] = out["sentence_type"].eq("control_weak_schema") if "sentence_type" in out.columns else False
    out["is_structured_json"] = out["parse_status"].eq("parsed") if "parse_status" in out.columns else False
    out["control_correct"] = out["is_control"] & out["literal_or_metaphorical"].eq("control") & out["main_image_schema"].eq("NONE")
    out["control_false_positive_schema"] = out["is_control"] & out["main_image_schema"].notna() & ~out["main_image_schema"].eq("NONE")
    out["schema_present_correct"] = out["schema_present"].eq(out["gold_schema_present"])
    out["primary_schema_correct"] = out["main_image_schema"].eq(out["expected_schema_primary"])
    out["lm_correct"] = out["literal_or_metaphorical"].eq(out["expected_literal_or_metaphorical"])
    return out

def summarize_group(df: pd.DataFrame, group_cols: list[str]) -> pd.DataFrame:
    if df.empty:
        return pd.DataFrame(columns=group_cols + [
            "n", "parse_rate", "schema_present_accuracy", "primary_schema_accuracy",
            "literal_metaphorical_accuracy", "control_accuracy",
            "control_false_positive_schema_rate", "non_control_lm_accuracy"
        ])
    rows = []
    for keys, g in df.groupby(group_cols, dropna=False):
        if not isinstance(keys, tuple):
            keys = (keys,)
        parsed = g[g["parse_status"].eq("parsed")] if "parse_status" in g.columns else g
        controls = parsed[parsed["is_control"]]
        non_controls = parsed[~parsed["is_control"]]
        row = dict(zip(group_cols, keys))
        row.update({
            "n": len(g),
            "parsed_n": len(parsed),
            "parse_rate": len(parsed) / len(g) if len(g) else None,
            "schema_present_accuracy": safe_rate(parsed["schema_present_correct"]) if len(parsed) else None,
            "primary_schema_accuracy": safe_accuracy(parsed, "main_image_schema", "expected_schema_primary"),
            "literal_metaphorical_accuracy": safe_accuracy(parsed, "literal_or_metaphorical", "expected_literal_or_metaphorical"),
            "control_accuracy": safe_rate(controls["control_correct"]) if len(controls) else None,
            "control_false_positive_schema_rate": safe_rate(controls["control_false_positive_schema"]) if len(controls) else None,
            "non_control_lm_accuracy": safe_rate(non_controls["lm_correct"]) if len(non_controls) else None,
        })
        rows.append(row)
    return pd.DataFrame(rows)

def display_percent_table(df: pd.DataFrame, percent_cols: list[str]) -> pd.DataFrame:
    out = df.copy()
    for col in percent_cols:
        if col in out.columns:
            out[col] = out[col].map(lambda x: pct(x) if x is not None else "NA")
    return out

In [2]:
parsed = add_derived_columns(read_jsonl(PARSED_PATH))
structured = parsed[parsed["parse_status"].eq("parsed")].copy()

print(f"Total records: {len(parsed)}")
print(f"Structured records: {len(structured)}")

Total records: 9000
Structured records: 7200


In [3]:
prompt_metrics = summarize_group(structured, ["prompt_id", "prompt_family", "prompt_generation"])
sentence_metrics = summarize_group(structured, ["prompt_id", "sentence_type"])
display(display_percent_table(prompt_metrics, [
    "schema_present_accuracy", "primary_schema_accuracy", "literal_metaphorical_accuracy",
    "control_accuracy", "control_false_positive_schema_rate", "non_control_lm_accuracy"
]))

,prompt_id,prompt_family,prompt_generation,n,parsed_n,parse_rate,schema_present_accuracy,primary_schema_accuracy,literal_metaphorical_accuracy,control_accuracy,control_false_positive_schema_rate,non_control_lm_accuracy
0,p_direct_schema_v1,direct_schema,v1,1800,1800,1.0,80.8%,71.9%,80.8%,41.9%,58.1%,99.9%
1,p_direct_schema_v2_abstention,direct_schema,v2_abstention,1800,1800,1.0,92.7%,85.9%,92.7%,78.5%,21.5%,99.8%
2,p_structured_roles_v1,structured_role_based,v1,1800,1800,1.0,90.7%,80.3%,90.5%,72.7%,27.3%,99.3%
3,p_structured_roles_v2_abstention,structured_role_based,v2_abstention,1800,1800,1.0,92.8%,84.3%,92.4%,78.3%,21.7%,99.4%


In [4]:
# Tutor-facing claims generated from current results.
def get_metric(df, prompt_id, metric):
    sub = df[df["prompt_id"].eq(prompt_id)]
    if sub.empty or metric not in sub.columns:
        return None
    return sub.iloc[0][metric]

rows = []
for prompt_id in sorted(structured["prompt_id"].dropna().unique()):
    rows.append({
        "prompt_id": prompt_id,
        "control_accuracy": get_metric(prompt_metrics, prompt_id, "control_accuracy"),
        "control_false_positive_schema_rate": get_metric(prompt_metrics, prompt_id, "control_false_positive_schema_rate"),
        "non_control_lm_accuracy": get_metric(prompt_metrics, prompt_id, "non_control_lm_accuracy"),
        "primary_schema_accuracy": get_metric(prompt_metrics, prompt_id, "primary_schema_accuracy"),
    })
tutor_table = pd.DataFrame(rows)
display(display_percent_table(tutor_table, [
    "control_accuracy", "control_false_positive_schema_rate", "non_control_lm_accuracy", "primary_schema_accuracy"
]))

,prompt_id,control_accuracy,control_false_positive_schema_rate,non_control_lm_accuracy,primary_schema_accuracy
0,p_direct_schema_v1,41.9%,58.1%,99.9%,71.9%
1,p_direct_schema_v2_abstention,78.5%,21.5%,99.8%,85.9%
2,p_structured_roles_v1,72.7%,27.3%,99.3%,80.3%
3,p_structured_roles_v2_abstention,78.3%,21.7%,99.4%,84.3%


In [5]:
scenario_rows = [
    {
        "research_issue": "Naïve prompt gives a good paraphrase",
        "what_it_means": "Ordinary meaning interpretation can be correct without explicit image-schema analysis.",
        "project_response": "Treat paraphrase adequacy and image-schema analysis as different semantic levels.",
        "evidence_to_report": "Naïve outputs are preserved as free_text_unparsed; structured prompts are evaluated separately."
    },
    {
        "research_issue": "Weak-schema controls perform poorly under v1",
        "what_it_means": "Image-schema prompts can induce over-attribution: the model finds a schema because the task asks for one.",
        "project_response": "Introduce a schema_present / NONE abstention gate.",
        "evidence_to_report": "Compare control false-positive schema rate in v1 vs v2 prompts."
    },
    {
        "research_issue": "v2 improves controls without hurting non-controls",
        "what_it_means": "The issue is prompt calibration, not necessarily inability to interpret weak-control sentences.",
        "project_response": "Frame v2 as a methodological improvement and a finding about abstention.",
        "evidence_to_report": "Report control_accuracy, false-positive rate, and non_control_lm_accuracy."
    },
    {
        "research_issue": "Literal and metaphorical examples both perform well",
        "what_it_means": "Models may reliably apply conventional image-schema categories in controlled examples.",
        "project_response": "Avoid claiming embodied cognition; claim recovery of image-schema-like representations under prompting.",
        "evidence_to_report": "Report sentence-type metrics and schema-family difficulty."
    },
    {
        "research_issue": "Schema-family differences",
        "what_it_means": "Some schemas may be more lexically visible or more conventionalised than others.",
        "project_response": "Analyse schema families separately rather than only overall accuracy.",
        "evidence_to_report": "Report accuracy by expected_schema_primary and confusion matrices."
    },
]
interpretation_matrix = pd.DataFrame(scenario_rows)
display(interpretation_matrix)

out = OUTPUTS_DIR / "tutor_response_interpretation_matrix_v2.csv"
interpretation_matrix.to_csv(out, index=False)
print(f"Wrote: {out}")

,research_issue,what_it_means,project_response,evidence_to_report
0,Naïve prompt gives a good paraphrase,Ordinary meaning interpretation can be correct...,Treat paraphrase adequacy and image-schema ana...,Naïve outputs are preserved as free_text_unpar...
1,Weak-schema controls perform poorly under v1,Image-schema prompts can induce over-attributi...,Introduce a schema_present / NONE abstention g...,Compare control false-positive schema rate in ...
2,v2 improves controls without hurting non-controls,"The issue is prompt calibration, not necessari...",Frame v2 as a methodological improvement and a...,"Report control_accuracy, false-positive rate, ..."
3,Literal and metaphorical examples both perform...,Models may reliably apply conventional image-s...,Avoid claiming embodied cognition; claim recov...,Report sentence-type metrics and schema-family...
4,Schema-family differences,Some schemas may be more lexically visible or ...,Analyse schema families separately rather than...,Report accuracy by expected_schema_primary and...


Wrote: /Users/Shared/image_schema_llm_project/data/outputs/tutor_response_interpretation_matrix_v2.csv


In [6]:
# Create a short markdown briefing that can be edited into an email or supervision note.
    briefing = f'''
# Tutor-facing results briefing

The project is not claiming that LLMs possess embodied cognition. It tests whether image-schema prompting provides an explicit, measurable intermediate representation beyond ordinary paraphrase.

## Key distinction

A naïve prompt may correctly paraphrase a sentence without exposing the conceptual structure being tested. The structured prompts instead ask the model to provide schema labels, schema presence, source/target domains, and schematic roles. This makes the output comparable across models, prompts, sentence types, and schema families.

## Current methodological finding

The weak-schema control condition is diagnostic. If a model assigns schemas to control sentences, this suggests prompt-induced over-attribution: the model is trying to satisfy the image-schema task even when the best answer is no clear schema.

The v2 prompt addresses this by introducing a schema-present / NONE abstention gate. The key test is whether v2 improves control accuracy and reduces false-positive schema assignments without damaging literal/metaphorical performance on non-control examples.

## Results to cite from notebook tables

Use:
- control_accuracy;
- control_false_positive_schema_rate;
- schema_present_accuracy;
- non_control_lm_accuracy;
- primary_schema_accuracy by schema family.

## Research value

The project therefore contributes not by proving embodied cognition in LLMs, but by testing whether a central construct from cognitive linguistics can be operationalised as an explainable annotation layer for LLM semantic interpretation.
'''
    out = OUTPUTS_DIR / "tutor_response_briefing_v2.md"
    out.write_text(briefing.strip(), encoding="utf-8")
    print(f"Wrote: {out}")
    print(briefing)

IndentationError: unexpected indent (109095446.py, line 2)

## Suggested supervision wording

The strongest phrasing is:

> The experiment distinguishes ordinary semantic adequacy from image-schema-structured interpretation. A naïve paraphrase may be correct, but it does not provide a comparable representation of schema presence, schema family, schematic roles, or source–target mappings. The weak-schema control condition is especially important because it tests whether image-schema prompting causes over-attribution. The v2 abstention prompt is therefore not just a technical fix; it operationalises the question of when an image-schema analysis is warranted at all.